In [ ]:
# ambiente local (celula 1 do notebook)

from dotenv import load_dotenv

load_dotenv("desenv.env", override=True)

from src.utils.gerenciador_sessao_spark_local import (
    GerenciadorSessaoSpark,
)

gerenciador_spark = GerenciadorSessaoSpark(
    nome_sessao="analise_transacoes",
    nome_arquivo_env_modelagem="desenv.env",
    exibir_configuracao=False,
)

spark = gerenciador_spark.criar_sessao_spark(
    db2=True
)

In [ ]:
# ambiente spark (celula 2 do notebook)

%run ./src/utils/gerenciador_sessao_spark_remoto.ipynb

In [ ]:
%%spark

import os

env_spark = dict(os.environ)

cliente_db2 = criar_cliente_db2_spark(
    env=env_spark
)

print("Conexão pronta para iniciar as queries.")

# MVP — transações correntes do cliente

A consulta usa exclusivamente `DB2GFP.TRAN_RLZD_INST_PCT`. Edite os três parâmetros na próxima célula antes de executar. `CODIGO_CLIENTE` é obrigatório e não possui valor padrão; as datas começam preenchidas com julho de 2026.

O intervalo é inclusivo, não pode ultrapassar 12 meses-calendário e deve estar integralmente nos 12 meses mais recentes em relação a `HOJE`. Somente transações efetivadas (`CD_EST_TRAN_INST = 0`) são retornadas.

# Estudo de validação


In [ ]:
%%spark
from datetime import date, datetime
CODIGO_CLIENTE = None
DATA_INICIO = '2026-07-01'
DATA_FIM = '2026-07-31'
if CODIGO_CLIENTE is None or DATA_INICIO is None or DATA_FIM is None: raise ValueError('Parâmetros obrigatórios ausentes.')
inicio=datetime.strptime(DATA_INICIO,'%Y-%m-%d').date(); fim=datetime.strptime(DATA_FIM,'%Y-%m-%d').date(); hoje=date.today()
if inicio>fim or (fim.year-inicio.year)*12+fim.month-inicio.month>12: raise ValueError('Intervalo inválido.')
if inicio<hoje.replace(year=hoje.year-1) or fim>hoje: raise ValueError('Período fora da janela móvel.')
print('Parâmetros válidos; CODIGO_CLIENTE é usado somente no filtro interno.')

## 1. Tabela principal
Começamos exclusivamente por DB2GFP.TRAN_RLZD_INST_PCT para confirmar moedas, valores, crédito/débito, instituição, marca, produto, tipo e categorias.

In [ ]:
%%spark
import hashlib
import traceback

_ultimo_diagnostico_jdbc = ''

def diagnosticar_jdbc_bruto(sql):
    global _ultimo_diagnostico_jdbc
    _ultimo_diagnostico_jdbc = ''
    print('### DIAGNÓSTICO JDBC BRUTO')
    try:
        reader = (cliente_db2.spark.read.format('jdbc')
            .option('url', cliente_db2.url)
            .option('driver', cliente_db2.driver)
            .option('user', cliente_db2.user)
            .option('password', cliente_db2.password)
            .option('dbtable', f'({sql}) DB2_DIAGNOSTICO')
            .option('fetchsize', 1)
            .option('queryTimeout', 60))
        reader.load().limit(1).count()
        print('A leitura JDBC bruta não reproduziu a falha.')
    except Exception as baixo:
        print(f'Tipo JDBC: {type(baixo).__module__}.{type(baixo).__name__}')
        print(f'Mensagem JDBC: {baixo!r}')
        _ultimo_diagnostico_jdbc = (f'Tipo JDBC: {type(baixo).__module__}.{type(baixo).__name__}\n'
            f'Mensagem JDBC: {baixo!r}\nTraceback JDBC original:\n{traceback.format_exc()}')
        print('Traceback JDBC original:')
        print(traceback.format_exc())
        for atributo in ('sqlstate', 'sqlcode', 'error_code', 'java_exception'):
            valor = getattr(baixo, atributo, None)
            if valor is not None:
                print(f'JDBC {atributo}: {valor!r}')

def consultar(sql):
    try:
        return cliente_db2.run_select(sql=sql, fetchsize=1000, query_timeout=300)
    except Exception as exc:
        print('### FALHA NA LEITURA DB2')
        print(f'Tipo da exceção: {type(exc).__module__}.{type(exc).__name__}')
        print(f'Mensagem original: {exc!r}')
        print(f'Tamanho do SQL: {len(sql)} caracteres')
        print(f'Identificador técnico do SQL: {hashlib.sha256(sql.encode("utf-8")).hexdigest()[:16]}')
        print('Traceback completo:')
        print(traceback.format_exc())
        causa = exc.__cause__ or exc.__context__
        if causa is not None:
            print('Causa encadeada:')
            print(''.join(traceback.format_exception(type(causa), causa, causa.__traceback__)))
        for atributo in ('sqlcode', 'sqlstate', 'error_code', 'response', 'details'):
            valor = getattr(exc, atributo, None)
            if valor is not None:
                print(f'{atributo}: {valor!r}')
        diagnosticar_jdbc_bruto(sql)
        raise RuntimeError(
            'Diagnóstico DB2 encapsulado:\n'
            f'wrapper={exc!r}\n'
            f'traceback_wrapper={traceback.format_exc()}\n'
            f'{_ultimo_diagnostico_jdbc}'
        ) from exc
base=f'''FROM DB2GFP.TRAN_RLZD_INST_PCT WHERE CD_CLI={CODIGO_CLIENTE} AND DT_TRAN BETWEEN DATE('{DATA_INICIO}') AND DATE('{DATA_FIM}') AND CD_EST_TRAN_INST=0'''
df_principal=consultar(f'''SELECT CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(VL_TRAN) VL_TOTAL,SUM(CASE WHEN CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN CD_NTZ_CTB_TRAN='C' THEN VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN CD_NTZ_CTB_TRAN='D' THEN VL_TRAN ELSE 0 END) VL_DEBITOS {base} GROUP BY CD_TIP_MOE_CRR''')
df_instituicoes=consultar(f'''SELECT CD_INST_PCT,CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(VL_TRAN) VL_TOTAL {base} GROUP BY CD_INST_PCT,CD_TIP_MOE_CRR''')
df_marcas=consultar(f'''SELECT NR_MCA_PCT_OPB,CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(VL_TRAN) VL_TOTAL {base} GROUP BY NR_MCA_PCT_OPB,CD_TIP_MOE_CRR''')
df_produtos=consultar(f'''SELECT CD_PRD,CD_TIP_TRAN,CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(VL_TRAN) VL_TOTAL,SUM(CASE WHEN NR_SEQL_CT_CLI IS NULL THEN 1 ELSE 0 END) QT_SEM_SEQUENCIA {base} GROUP BY CD_PRD,CD_TIP_TRAN,CD_TIP_MOE_CRR''')
df_integridade=consultar(f'''SELECT COUNT(*) QT_REGISTROS,COUNT(DISTINCT NR_TRAN_INST_PCT) QT_TRANSACOES,MIN(DT_TRAN) DT_MIN,MAX(DT_TRAN) DT_MAX,SUM(CASE WHEN CD_CTGR_TRAN IS NULL THEN 1 ELSE 0 END) QT_SEM_CATEGORIA {base}''')
for nome,frame in [('Resumo por moeda',df_principal),('Instituições',df_instituicoes),('Marcas',df_marcas),('Produtos e tipos',df_produtos),('Integridade',df_integridade)]: print('\n### '+nome); frame.show(truncate=False)
print('CONFIRMADO NOS DADOS quando campos e códigos observados forem consistentes; caso contrário, CONFIRMADO COM LIMITAÇÃO.')

## 2. Lacuna concreta — contas Open Finance
A tabela principal não possui o universo de contas nem estados de autorização. INF_OPB_CT_CLI entra somente para essa lacuna. Conta autorizada exige NR_SEQL_AUTZ_OPB preenchido e IN_AUTZ_EXB_TRAN='S'.

In [ ]:
%%spark
df_contas=consultar(f'''SELECT CD_INST_PCT,CD_TIP_CT_OPB,CD_PRD,CD_MDLD_PRD,TRIM(IN_AUTZ_EXB_TRAN) IN_AUTZ_EXB_TRAN,TRIM(IN_EXB_DADO_CT) IN_EXB_DADO_CT,CD_EST_RCS_OPB,COUNT(*) QT_RELACIONAMENTOS FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE} GROUP BY CD_INST_PCT,CD_TIP_CT_OPB,CD_PRD,CD_MDLD_PRD,TRIM(IN_AUTZ_EXB_TRAN),TRIM(IN_EXB_DADO_CT),CD_EST_RCS_OPB''')
df_flags=consultar(f'''SELECT TRIM(IN_AUTZ_EXB_TRAN) IN_AUTZ_EXB_TRAN,COUNT(*) QT FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE} GROUP BY TRIM(IN_AUTZ_EXB_TRAN)''')
df_chaves=consultar(f'''SELECT NR_MCA_PCT_OPB,NR_SEQL_CT_CLI,COUNT(*) QT FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE} GROUP BY NR_MCA_PCT_OPB,NR_SEQL_CT_CLI HAVING COUNT(*)>1''')
df_contas.show(truncate=False); df_flags.show(truncate=False); print('Chaves duplicadas:',df_chaves.count())
print('Classificação: CONFIRMADO NOS DADOS se valores S/N e chaves forem únicos; caso contrário, NÃO CONFIRMADO.')

In [ ]:
%%spark
df_estados=consultar(f'''SELECT CASE WHEN NR_SEQL_AUTZ_OPB IS NULL THEN 'SEM_RELACIONAMENTO_OF' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' AND CD_EST_RCS_OPB=2 THEN 'AUTORIZADA_DISPONIVEL' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' AND TRIM(IN_EXB_DADO_CT)='N' THEN 'AUTORIZADA_OCULTA' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' THEN 'AUTORIZADA' WHEN TRIM(IN_AUTZ_EXB_TRAN)='N' THEN 'NAO_AUTORIZADA' ELSE 'ESTADO_NAO_RECONHECIDO' END STATUS_CONTA,COUNT(*) QT FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE} GROUP BY CASE WHEN NR_SEQL_AUTZ_OPB IS NULL THEN 'SEM_RELACIONAMENTO_OF' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' AND CD_EST_RCS_OPB=2 THEN 'AUTORIZADA_DISPONIVEL' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' AND TRIM(IN_EXB_DADO_CT)='N' THEN 'AUTORIZADA_OCULTA' WHEN TRIM(IN_AUTZ_EXB_TRAN)='S' THEN 'AUTORIZADA' WHEN TRIM(IN_AUTZ_EXB_TRAN)='N' THEN 'NAO_AUTORIZADA' ELSE 'ESTADO_NAO_RECONHECIDO' END''')
df_estados.show(truncate=False); print('Estados são atributos observados; não reconstruímos histórico de consentimento.')

In [ ]:
%%spark
contas_sql=f'''(SELECT CD_CLI_TITR_CT,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI,CD_INST_PCT,CD_TIP_CT_OPB,CD_PRD,CASE WHEN NR_SEQL_AUTZ_OPB IS NOT NULL AND TRIM(IN_AUTZ_EXB_TRAN)='S' THEN 1 ELSE 0 END AUTORIZADA,ROW_NUMBER() OVER(ORDER BY CD_INST_PCT,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI) CONTA_ESTUDO FROM DB2GFP.INF_OPB_CT_CLI WHERE CD_CLI_TITR_CT={CODIGO_CLIENTE})'''
transacoes_sql=f'''(SELECT NR_TRAN_INST_PCT,CD_CLI,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI,CD_TIP_MOE_CRR,VL_TRAN,CD_NTZ_CTB_TRAN,CD_PRD,CD_TIP_TRAN,CD_CTGR_TRAN FROM DB2GFP.TRAN_RLZD_INST_PCT WHERE CD_CLI={CODIGO_CLIENTE} AND DT_TRAN BETWEEN DATE('{DATA_INICIO}') AND DATE('{DATA_FIM}') AND CD_EST_TRAN_INST=0)'''
classificacao='CASE WHEN b.NR_SEQL_CT_CLI IS NULL THEN \'SEM_SEQUENCIA\' WHEN r.CD_CLI_TITR_CT IS NULL THEN \'SEM_RELACIONAMENTO\' WHEN r.AUTORIZADA=1 THEN \'CONTA_AUTORIZADA\' ELSE \'CONTA_NAO_AUTORIZADA\' END'
df_associacao=consultar(f'''SELECT b.CD_TIP_MOE_CRR,{classificacao} CLASSIFICACAO,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN b.VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN b.VL_TRAN ELSE 0 END) VL_DEBITOS FROM {transacoes_sql} b LEFT JOIN {contas_sql} r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI GROUP BY b.CD_TIP_MOE_CRR,{classificacao}''')
df_linhas=consultar(f'''SELECT COUNT(*) QT_LINHAS,COUNT(DISTINCT b.NR_TRAN_INST_PCT) QT_TRANSACOES FROM {transacoes_sql} b LEFT JOIN {contas_sql} r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI''')
df_contas_sem_mov=consultar(f'''SELECT COUNT(*) QT_CONTAS_AUTORIZADAS_SEM_MOV FROM (SELECT DISTINCT CONTA_ESTUDO,CD_CLI_TITR_CT,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI FROM {contas_sql} WHERE AUTORIZADA=1) r LEFT JOIN (SELECT DISTINCT CD_CLI,NR_MCA_PCT_OPB,NR_SEQL_CT_CLI FROM {transacoes_sql}) b ON b.CD_CLI=r.CD_CLI_TITR_CT AND b.NR_MCA_PCT_OPB=r.NR_MCA_PCT_OPB AND b.NR_SEQL_CT_CLI=r.NR_SEQL_CT_CLI WHERE b.CD_CLI IS NULL''')
df_associacao.show(truncate=False); df_linhas.show(truncate=False); df_contas_sem_mov.show(truncate=False)
print('CONFIRMADO NOS DADOS se QT_LINHAS=QT_TRANSACOES, chaves únicas e cobertura quantificada; caso contrário, CONFIRMADO COM LIMITAÇÃO ou NÃO CONFIRMADO.')

## 3. Resumo por conta

Contas autorizadas recebem rotulos temporarios nao identificadores. Totais, creditos e debitos ficam separados por moeda. Produtos e tipos permanecem como agregados, sem exibicao de agencia, conta, cartao ou transacao individual.


In [ ]:
%%spark
df_contas_resumo=consultar(f'''SELECT r.CONTA_ESTUDO,r.CD_INST_PCT,r.CD_TIP_CT_OPB,r.CD_PRD,b.CD_TIP_MOE_CRR,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN b.VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN b.VL_TRAN ELSE 0 END) VL_DEBITOS FROM {transacoes_sql} b JOIN {contas_sql} r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI AND r.AUTORIZADA=1 GROUP BY r.CONTA_ESTUDO,r.CD_INST_PCT,r.CD_TIP_CT_OPB,r.CD_PRD,b.CD_TIP_MOE_CRR''')
df_contas_tipos=consultar(f'''SELECT r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,b.CD_PRD,b.CD_TIP_TRAN,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL FROM {transacoes_sql} b JOIN {contas_sql} r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI AND r.AUTORIZADA=1 GROUP BY r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,b.CD_PRD,b.CD_TIP_TRAN''')
print('### Resumo financeiro por conta autorizada')
df_contas_resumo.show(truncate=False)
print('### Produtos e tipos por conta autorizada')
df_contas_tipos.show(truncate=False)
print('CONFIRMADO NOS DADOS quando a associacao autorizada estiver segura; contas sem movimento permanecem no estudo de cobertura.')


## 4. Grupos e categorias por conta

A categoria atual e a referencia principal. Antes de traduzir codigos, o notebook valida unicidade dos dominios. Se categoria ou grupo estiver duplicado no dominio, a sumarizacao interpretada deve ser bloqueada. Categorias originais permanecem separadas; codigos sem correspondencia ficam em buckets explicitos.


In [ ]:
%%spark
df_cat_dups=consultar('SELECT CD_CTGR_TRAN,COUNT(*) QT FROM DB2GFP.CTGR_TRAN_OPB GROUP BY CD_CTGR_TRAN HAVING COUNT(*)>1')
df_group_dups=consultar('SELECT CD_GR_CTGR_TRAN,COUNT(*) QT FROM DB2GFP.GR_CTGR_TRAN GROUP BY CD_GR_CTGR_TRAN HAVING COUNT(*)>1')
dominios_unicos=(df_cat_dups.count()==0 and df_group_dups.count()==0)
print('Dominios sem duplicidade:',dominios_unicos)
if not dominios_unicos: raise ValueError('NAO CONFIRMADO: dominio de categoria ou grupo duplicado; sumarizacao interpretada bloqueada.')
df_grupos=consultar(f'''SELECT r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,COALESCE(g.CD_GR_CTGR_TRAN,'SEM_GRUPO') CD_GRUPO,COALESCE(g.TX_DCR_GR_CTGR,'Sem grupo') DS_GRUPO,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN b.VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN b.VL_TRAN ELSE 0 END) VL_DEBITOS FROM {transacoes_sql} b JOIN {contas_sql} r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI AND r.AUTORIZADA=1 LEFT JOIN DB2GFP.CTGR_TRAN_OPB c ON c.CD_CTGR_TRAN=b.CD_CTGR_TRAN LEFT JOIN DB2GFP.GR_CTGR_TRAN g ON g.CD_GR_CTGR_TRAN=c.CD_GR_CTGR_TRAN GROUP BY r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,g.CD_GR_CTGR_TRAN,g.TX_DCR_GR_CTGR''')
df_categorias=consultar(f'''SELECT r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,COALESCE(c.CD_CTGR_TRAN,'SEM_CATEGORIA') CD_CATEGORIA,COALESCE(c.TX_DCR_CTGR_TRAN,'Sem categoria') DS_CATEGORIA,COALESCE(c.CD_GR_CTGR_TRAN,'SEM_GRUPO') CD_GRUPO,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN b.VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN b.VL_TRAN ELSE 0 END) VL_DEBITOS FROM {transacoes_sql} b JOIN {contas_sql} r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI AND r.AUTORIZADA=1 LEFT JOIN DB2GFP.CTGR_TRAN_OPB c ON c.CD_CTGR_TRAN=b.CD_CTGR_TRAN GROUP BY r.CONTA_ESTUDO,b.CD_TIP_MOE_CRR,c.CD_CTGR_TRAN,c.TX_DCR_CTGR_TRAN,c.CD_GR_CTGR_TRAN''')
print('### Grupos por conta autorizada')
df_grupos.show(truncate=False)
print('### Categorias por conta autorizada')
df_categorias.show(truncate=False)
print('CONFIRMADO NOS DADOS se dominios unicos e cobertura fecharem; duplicidades tornam a interpretacao NAO CONFIRMADA.')


## 5. Reconciliacao e conclusao

Por moeda, cliente = contas autorizadas + classes nao associadas. Grupos/categorias conhecidos + buckets sem classificacao devem representar todas as transacoes autorizadas.


In [ ]:
%%spark
df_reconciliacao=consultar(f'''SELECT b.CD_TIP_MOE_CRR,'CLIENTE' ESCOPO,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN b.VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN b.VL_TRAN ELSE 0 END) VL_DEBITOS FROM {transacoes_sql} b GROUP BY b.CD_TIP_MOE_CRR UNION ALL SELECT b.CD_TIP_MOE_CRR,CASE WHEN r.AUTORIZADA=1 THEN 'CONTAS_AUTORIZADAS' ELSE 'NAO_ASSOCIADA' END ESCOPO,COUNT(*) QT_TRANSACOES,SUM(b.VL_TRAN) VL_TOTAL,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN 1 ELSE 0 END) QT_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='C' THEN b.VL_TRAN ELSE 0 END) VL_CREDITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN 1 ELSE 0 END) QT_DEBITOS,SUM(CASE WHEN b.CD_NTZ_CTB_TRAN='D' THEN b.VL_TRAN ELSE 0 END) VL_DEBITOS FROM {transacoes_sql} b LEFT JOIN {contas_sql} r ON r.CD_CLI_TITR_CT=b.CD_CLI AND r.NR_MCA_PCT_OPB=b.NR_MCA_PCT_OPB AND r.NR_SEQL_CT_CLI=b.NR_SEQL_CT_CLI GROUP BY b.CD_TIP_MOE_CRR,CASE WHEN r.AUTORIZADA=1 THEN 'CONTAS_AUTORIZADAS' ELSE 'NAO_ASSOCIADA' END''')
df_reconciliacao.show(truncate=False)
print('CONFIRMADO NOS DADOS se os totais por moeda fecharem; diferencas ficam explicadas por nao associacao ou marcadas como inesperadas.')
print('Menor conjunto esperado: TRAN_RLZD_INST_PCT; INF_OPB_CT_CLI para contas; CTGR_TRAN_OPB e GR_CTGR_TRAN somente se descricoes unicas forem necessarias.')
print('Nenhuma fonte externa de autorizacao participa do estudo ou da solucao.')


## 6. Diagnostico auxiliar das tabelas

Esta secao fica no final porque nao define a visao final. Ela serve para consultar o diagnostico automatico das tabelas candidatas, quando for necessario conferir metadados, tipos, dominios pequenos, colunas temporais e sinais de cardinalidade.

O diagnostico deve ser usado como apoio para interpretar as validacoes anteriores. Ele nao substitui a regra do estudo: a tabela so entra na arquitetura minima se resolver uma lacuna concreta da visao sumarizada.


In [ ]:
%%spark
# Diagnostico auxiliar das tabelas do estudo.
#
# Como pode consultar catalogo e amostras das tabelas, ele fica desativado por padrao.
# Para executar, altere EXECUTAR_DIAGNOSTICO_TABELAS para True e rode esta celula.
# Os resultados devem ser lidos como apoio documental, nao como produto final da visao.

EXECUTAR_DIAGNOSTICO_TABELAS = True

tabelas_diagnostico = [
    ('DB2GFP', 'TRAN_RLZD_INST_PCT', DATA_INICIO),
    ('DB2GFP', 'INF_OPB_CT_CLI', None),
    ('DB2GFP', 'CTGR_TRAN_OPB', None),
    ('DB2GFP', 'GR_CTGR_TRAN', None),
]

diagnosticos_tabelas = {}

if EXECUTAR_DIAGNOSTICO_TABELAS:
    for schema, nome_tabela, data_inicio_diagnostico in tabelas_diagnostico:
        print(f'\n### Diagnostico: {schema}.{nome_tabela}')
        diagnosticos_tabelas[nome_tabela] = cliente_db2.diagnosticar_tabela(
            schema=schema,
            nome_tabela=nome_tabela,
            data_inicio=data_inicio_diagnostico,
            show=True,
            truncate=False,
        )
else:
    print('Diagnostico de tabelas preparado, mas nao executado.')
    print('Para rodar, altere EXECUTAR_DIAGNOSTICO_TABELAS para True nesta celula.')


## Encerramento do estudo

Depois da execucao das validacoes e, se necessario, do diagnostico auxiliar, a conclusao deve registrar o menor conjunto de tabelas comprovado pelos dados reais. A rotina principal so deve receber aquilo que foi confirmado neste notebook e aprovado para a visao final.